# Required Imports

In [1]:
import os
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models

import numpy as np
from numpy import dot
from numpy.linalg import norm

from tqdm import tqdm

# Imports PIL module
from PIL import Image

## Parameters

In [2]:
# Set a fixed random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)  # Seed for NumPy random number generator
torch.manual_seed(RANDOM_SEED)  # Seed for PyTorch random number generator

# Set the depth of the network
depth = 32  # Initial depth of the Siamese network
img_channels = 3  # Number of color channels (3 for RGB)
img_w = 172  # Image width
img_h = 128  # Image height

# Determine the computing device (GPU if available, otherwise CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f'device being used: {device}')


# Configuration settings
n_classes = 40 # Number of identities in the training set
n_epochs = 50  # Number of training epochs
train_batch_size = 32  # Batch size for training
val_batch_size = 1  # Batch size for validation
test_batch_size = 1  # Batch size for testing
learning_rate = 0.0005  # Learning rate for the optimizer
backbone = 'siameseNet'  # Backbone network architecture ['siameseNet', 'VGG16', 'VGG19', 'denseNet', 'MobileNetV3', 'efficientNetB0', 'ViT16']

device being used: mps


## Load Probe and Gallery sets

In [3]:
# Define ID
def extract_id_from_imagename(filename):
    """ Extract the base ID from the filename (e.g., '00001m' from '00001m_0.png') """
    match = re.search(r'(\d+[mf])\_', filename) 
    return match.group(1) if match else None

# Define a custom dataset class for training, specific to handling probe and gallery images
class ProbeGalleryTrain(Dataset):
    def __init__(self, data_root, transform=None):
        self.data_root = data_root  
        self.data = self.read_folder()  
        self.samples = self.create_samples()  
        self.transform = transform  

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        im1_pth, im2_pth, is_match, probe_target, gallery_target = self.samples[idx]  
        im1 = Image.open(im1_pth)  
        im2 = Image.open(im2_pth)  

        if self.transform:
            im1 = self.transform(im1)
            im2 = self.transform(im2)

        return [im1, im2, is_match, probe_target, gallery_target]

    def read_folder(self):
        paths = []  
        labels = []  
        for dirpath, dirnames, filenames in os.walk(self.data_root):
            files = [f for f in filenames if f.split('.')[-1] in ['jpg', 'jpeg', 'png']]
            for item in files:
                label = dirpath.split('/')[-1]
                paths.append([os.path.join(dirpath, item), label])
                if label not in labels:
                    labels.append(label)
        return paths

    def create_samples(self):
        probes = [x[0] for x in self.data if x[1] == 'Probe']  
        gallery = [x[0] for x in self.data if x[1] == 'Gallery']  

        # Crea una mappatura: ID originale -> ID progressivo [0, 39]
        unique_ids = sorted(list(set([extract_id_from_imagename(p) for p in probes if extract_id_from_imagename(p)])))
        id_to_class = {original_id: idx for idx, original_id in enumerate(unique_ids)}

        samples = []
        for probe in probes:
            probe_id = extract_id_from_imagename(probe)
            if not probe_id: continue
            probe_class = id_to_class[probe_id] 
            
            for item in gallery:
                item_id = extract_id_from_imagename(item)
                if not item_id: continue
                item_class = id_to_class[item_id]
                
                is_match = 1 if probe_class == item_class else -1
                samples.append([probe, item, is_match, probe_class, item_class])

        return samples

# create the val/test dataset
class ProbeGalleryValTest(Dataset):
    def __init__(self, data_root, transform=None):
        self.data_root = data_root  
        self.data = self.read_folder()  
        self.samples = self.create_samples()  
        self.transform = transform  

    def __len__(self):
        return len(self.samples)  

    def __getitem__(self, idx):
      probe_pth, gallery_pths, is_match, probe_target, gallery_targets = self.samples[idx]  
      probe = Image.open(probe_pth)  

      gallery = []
      for item in gallery_pths:
          gallery.append(Image.open(item))

      if self.transform:
          probe = self.transform(probe)
          for i in range(len(gallery)):
              gallery[i] = self.transform(gallery[i])

      return [probe, gallery, is_match, probe_pth, gallery_pths, probe_target, gallery_targets]

    def read_folder(self):
        paths = []  
        labels = []  
        for dirpath, dirnames, filenames in os.walk(self.data_root):
            files = [f for f in filenames if f.split('.')[-1] in ['jpg', 'jpeg', 'png']]
            for item in files:
                label = dirpath.split('/')[-1]  
                if label not in labels:
                    labels.append(label)
                paths.append([f'{dirpath}/{item}', label])  
        return paths

    def create_samples(self):
        probes = [x[0] for x in self.data if x[1] == 'Probe']  
        gallery = [x[0] for x in self.data if x[1] == 'Gallery']  

        samples = []
        for probe in probes:
          probe_id = extract_id_from_imagename(probe)  
          if not probe_id: continue

          is_match = []
          gallery_targets = []
          
          for item in gallery:
            item_id = extract_id_from_imagename(item)  
            if not item_id: continue
            
            is_match.append(1 if probe_id == item_id else -1)
            gallery_targets.append(item_id)

          samples.append([probe, gallery, is_match, probe_id, gallery_targets])
        return samples

In [4]:
# Define the transformation to be applied to the images (converting them to PyTorch tensors)
if backbone == 'ViT16':
    transform = transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
    ])
else:
    transform = transforms.Compose([
        transforms.Resize(128), 
        transforms.ToTensor(),
    ])

# Load the training, validation, and test datasets from specified directories and apply the defined transformation
train_dataset = ProbeGalleryTrain('./output/ImageData/D1/', transform=transform)
train_dataloader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)

test_dataset = ProbeGalleryValTest('./output/ImageData/D2t/', transform=transform)
test_dataloader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False)



# Neural Network Models

In [5]:
# Define a Siamese network class inheriting from nn.Module
class BSN(nn.Module):
    def __init__(self, backbone, img_channels, depth, img_w, img_h, n_classes):
        super(BSN, self).__init__()

        self.backbone = backbone
        
        if self.backbone == 'siameseNet':
            # Initialize convolutional and pooling layers
            # First convolutional layer with specified input channels and depth
            self.conv1 = nn.Conv2d(img_channels, depth, kernel_size=3, padding=1)
            # First pooling layer, reduces spatial dimensions by half
            self.pool1 = nn.MaxPool2d(2, 2)
            # Second convolutional layer, doubling the depth
            self.conv2 = nn.Conv2d(depth, depth*2, kernel_size=3, padding=1)
            # Second pooling layer, further reducing spatial dimensions
            self.pool2 = nn.MaxPool2d(2, 2)

            # Calculate the output size after the convolutional and pooling layers
            output_size = depth*2 * img_h//4 * img_w//4
            
            

        if self.backbone == 'VGG19':
            self.model = models.vgg19_bn().features # VGG19 with batch normalization

            # Create a dummy input tensor of the correct size
            dummy_input = torch.randn(1, img_channels, img_h, img_w)

            # Run a forward pass through the convolutional layers
            output = self.model(dummy_input)

            # Calculate the output size
            output_size = output.view(output.size(0), -1).size(1)
        
        if self.backbone == 'VGG16':
            self.model = models.vgg16_bn().features # VGG16 with batch normalization

            # Create a dummy input tensor of the correct size
            dummy_input = torch.randn(1, img_channels, img_h, img_w)

            # Run a forward pass through the convolutional layers
            output = self.model(dummy_input)

            # Calculate the output size
            output_size = output.view(output.size(0), -1).size(1)

        if self.backbone == 'denseNet':
            self.model = models.densenet121().features # DenseNet121 with batch normalization
            
            # Create a dummy input tensor of the correct size
            dummy_input = torch.randn(1, img_channels, img_h, img_w)

            # Run a forward pass through the convolutional layers
            output = self.model(dummy_input)

            # Calculate the output size
            output_size = output.view(output.size(0), -1).size(1)

        if self.backbone == "MobileNetV3":
            self.model = models.mobilenet_v3_small().features # MobileNetV3 with batch normalization

            # Create a dummy input tensor of the correct size
            dummy_input = torch.randn(1, img_channels, img_h, img_w)

            # Run a forward pass through the convolutional layers
            output = self.model(dummy_input)

            # Calculate the output size
            output_size = output.view(output.size(0), -1).size(1)
            
        if self.backbone == 'efficientNetB0':
            self.model = models.efficientnet_b0().features # VGG16 with batch normalization

            # Create a dummy input tensor of the correct size
            dummy_input = torch.randn(1, img_channels, img_h, img_w)

            # Run a forward pass through the convolutional layers
            output = self.model(dummy_input)

            # Calculate the output size
            output_size = output.view(output.size(0), -1).size(1)

        if self.backbone == "ViT16":
            self.model = models.vit_b_16() # ViT16
            self.model.heads = nn.Identity() # remove the classification head

            # Create a dummy input tensor of the correct size
            dummy_input = torch.randn(1, img_channels, 224, 224)

            # Run a forward pass through the convolutional layers
            output = self.model(dummy_input)

            # Calculate the output size
            output_size = output.view(output.size(0), -1).size(1)
        
        self.fc1 = nn.Linear(output_size, 256)
        self.fc2 = nn.Linear(256, 256)
        self.classifier = nn.Linear(256,n_classes)
        self.dropout = nn.Dropout()

    # Define the forward pass for one branch of the Siamese network
    def forward_one(self, x):
        if self.backbone == 'siameseNet':
            # Apply the first convolutional layer followed by ReLU activation and pooling
            x = self.pool1(F.relu(self.conv1(x)))
            # Apply the second convolutional layer, ReLU, and pooling
            x = self.pool2(F.relu(self.conv2(x)))
            # Flatten the output for the fully connected layer
            x = x.view(x.size(0), -1)
            # Apply the first fully connected layer with ReLU activation
            x = self.dropout(F.relu(self.fc1(x)))
            # Apply the second fully connected layer
            x = self.dropout(F.relu(self.fc2(x)))
            y = self.classifier(x)
        else:
            # Apply the selected model to the input
            x = self.model(x)
            # flatten output 
            x = x.view(x.size(0), -1)
            # Apply classifier component
            x = self.dropout(F.relu(self.fc1(x)))
            x = self.dropout(F.relu(self.fc2(x)))
            y = self.classifier(x)
        
        return x, y

    # Define the forward pass for the whole Siamese network
    def forward(self, input1, input2):
        # Process each input through the network
        output1, output_class1 = self.forward_one(input1)
        output2, output_class2= self.forward_one(input2)
        return output1, output_class1, output2, output_class2

# Training and Testing functions

In [6]:
def compute_ap(matches):
    """Compute the average precision (AP) given ranks of positive images and the number of positive images.
    Args:
        matches (list): The ranks of positive images sorted in ascending order.
    Returns:
        float: The average precision for the given data.
    """
    if len(matches) == 0:
        return 0.0
    
    matched = 0
    ap = 0.0
    
    for rank, match in enumerate(matches, start=1):
        if match:
            matched += 1
            ap += matched / rank

    return ap / matched

def testing_step(model, test_dataloader):
    # Switch the model to evaluation mode. This turns off specific layers/features like dropout.
    model.eval()

    # Initialize counters for correct predictions at different ranks
    correct1, correct2, correct3, correct4, correct5 = 0, 0, 0, 0, 0
    correct6, correct7, correct8, correct9, correct10 = 0, 0, 0, 0, 0
    samples = 0  
    
    # Set the rank limit for evaluation
    K = 10
    AP_scores = []  
    rank_matrix = []

    # Disable gradient calculation for efficiency and to prevent changes to the model
    with torch.no_grad():
        # Iterate over the test dataloader
        for data in tqdm(test_dataloader, leave=False, total=len(test_dataloader)):
            probe, gallery, is_match, probe_pth, gallery_pths, probe_target, gallery_targets = data

            scores = []
            gallery = torch.cat(gallery, axis=0)
            probe_batch = probe.repeat(gallery.shape[0], 1, 1, 1)

            output1, output_class1, output2, output_class2 = model(probe_batch.to(device), gallery.to(device))

            for i in range(len(probe_batch)):
                a = output1[i].detach().cpu().numpy()
                b = output2[i].detach().cpu().numpy()
                cos_sim = dot(a, b)/(norm(a)*norm(b))
                scores.append([cos_sim, gallery_pths[i][0]])
            
            samples += 1

            topk = sorted(scores, key=lambda element: (element[0]), reverse=True)
            rank_matrix.append([probe_pth[0], topk])

            # -- CALCOLO mAP --
            pos_ranks = []
            probe_id = extract_id_from_imagename(probe_pth[0])
            
            for i, score in enumerate(topk):
                gallery_id = extract_id_from_imagename(score[1])
                if probe_id == gallery_id:
                    pos_ranks.append(i)
                    
            if pos_ranks:
                AP = compute_ap(pos_ranks)  
                AP_scores.append(AP)  

        # -- CALCOLO MATRICE DEI RANK --
        for idx, probe in enumerate(rank_matrix):
            probe_id = extract_id_from_imagename(probe[0])
            
            for i in range(K):
                gallery_id = extract_id_from_imagename(probe[1][i][1])
                
                if probe_id == gallery_id:
                    if i == 0:
                        correct1 += 1; correct2 += 1; correct3 += 1; correct4 += 1; correct5 += 1
                        correct6 += 1; correct7 += 1; correct8 += 1; correct9 += 1; correct10 += 1
                    elif i == 1:
                        correct2 += 1; correct3 += 1; correct4 += 1; correct5 += 1; correct6 += 1
                        correct7 += 1; correct8 += 1; correct9 += 1; correct10 += 1
                    elif i == 2:
                        correct3 += 1; correct4 += 1; correct5 += 1; correct6 += 1; correct7 += 1
                        correct8 += 1; correct9 += 1; correct10 += 1
                    elif i == 3:
                        correct4 += 1; correct5 += 1; correct6 += 1; correct7 += 1; correct8 += 1
                        correct9 += 1; correct10 += 1
                    elif i == 4:
                        correct5 += 1; correct6 += 1; correct7 += 1; correct8 += 1; correct9 += 1
                        correct10 += 1
                    elif i == 5:
                        correct6 += 1; correct7 += 1; correct8 += 1; correct9 += 1; correct10 += 1
                    elif i == 6:
                        correct7 += 1; correct8 += 1; correct9 += 1; correct10 += 1
                    elif i == 7:
                        correct8 += 1; correct9 += 1; correct10 += 1
                    elif i == 8:
                        correct9 += 1; correct10 += 1
                    elif i == 9:
                        correct10 += 1
                    break

    mAP = np.mean(AP_scores)
    print(f'Total samples: {samples}, Rank #1: {correct1/samples*100.0:.3f}%, Rank #2: {correct2/samples*100.0:.3f}%, Rank #3: {correct3/samples*100.0:.3f}%, Rank #4: {correct4/samples*100.0:.3f}%, Rank #5: {correct5/samples*100.0:.3f}%, Rank #6: {correct6/samples*100.0:.3f}%, Rank #7: {correct7/samples*100.0:.3f}%, Rank #8: {correct8/samples*100.0:.3f}%, Rank #9: {correct9/samples*100.0:.3f}%, Rank #10: {correct10/samples*100.0:.3f}%, mAP:{mAP*100.0:.3f}%')
    return correct1/samples, correct5/samples, correct10/samples, mAP

In [7]:
def training_model(model, train_dataloader, test_dataloader, optimizer, embg_criterion, cat_criterion, n_epochs):
    # Initialize variable to store the best rank #10 accuracy
    bestRank1 = 0.0
    bestRank5 = 0.0
    bestRank10 = 0.0

    # Training loop for a specified number of epochs
    for epoch in range(n_epochs):
        # Set the network in training mode (this enables features like dropout and batch normalization)
        model.train()

        # Initialize a variable to accumulate loss over the epoch
        train_loss = 0.0

        # Iterate over batches in the training data loader
        for batch in tqdm(train_dataloader, leave=False, total=len(train_dataloader)):
            # Unpack the batch into images and target values and move them to the computation device
            img1 = batch[0].to(device)
            img2 = batch[1].to(device)
            is_match = batch[2].to(device)
            img1_target = batch[3].to(device)
            img2_target = batch[4].to(device)
            
            # Forward pass: Compute the output of the network for both images in the pair
            output1, output_class1, output2, output_class2 = model(img1, img2)

            # Compute the loss based on the outputs and the target values
            embg_loss = embg_criterion(output1, output2, is_match)
            cat_loss = cat_criterion(output_class1, img1_target) + cat_criterion(output_class2, img2_target)
            loss = embg_loss + cat_loss

            # Zero the gradients before running the backward pass
            optimizer.zero_grad()

            # Backward pass: Compute gradient of the loss with respect to network parameters
            loss.backward()

            # Perform a single optimization step (parameter update)
            optimizer.step()

            # Accumulate the loss over the batches
            train_loss += loss.item()

        # Perform validation step and return the rank #10 accuracy for this epoch
        rank1, rank5, rank10, _ = testing_step(model, test_dataloader)

        # Folder to save model weights
        models_folder = os.path.join('./output/models/')
        if not os.path.exists(models_folder):
            os.makedirs(models_folder, exist_ok=True)

        # Check if the current model is the best so far; if so, save it and update the best rank #1 accuracy
        if rank1 > bestRank1:
            torch.save(model.state_dict(), f'{models_folder}/best_{backbone}_state_r1.bin')
            bestRank1 = rank1

        torch.save(model.state_dict(), f'{models_folder}/best_{backbone}_epoch_'+str(epoch)+'.bin')

        # Compute the average loss for this epoch and print it
        running_loss = train_loss / len(train_dataloader)
        print(f"Epoch: [{epoch+1}/{n_epochs}]| Loss: {running_loss:.5f}")

    return None

## Model Parameters

In [8]:
# Initialize the Siamese network with the specified parameters and move it to the selected device
model = BSN(backbone, img_channels, depth, img_w, img_h, n_classes).to(device)

# Define the loss function for training the network (Cosine Embedding Loss)
embg_criterion = nn.CosineEmbeddingLoss()
cat_criterion = nn.CrossEntropyLoss()

# Define the optimizer for training, using the Adam algorithm with a learning rate of 0.0005
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

## Network Training

In [9]:
# Start training model
training_model(model, train_dataloader, test_dataloader, optimizer, embg_criterion, cat_criterion, n_epochs)

Total samples: 410, Rank #1: 64.634%, Rank #2: 72.439%, Rank #3: 76.585%, Rank #4: 79.268%, Rank #5: 80.732%, Rank #6: 82.683%, Rank #7: 84.878%, Rank #8: 86.341%, Rank #9: 87.073%, Rank #10: 87.805%, mAP:86.147%
Epoch: [1/50]| Loss: 0.70656


Total samples: 410, Rank #1: 56.341%, Rank #2: 68.293%, Rank #3: 74.146%, Rank #4: 78.293%, Rank #5: 80.000%, Rank #6: 81.220%, Rank #7: 82.195%, Rank #8: 83.902%, Rank #9: 85.610%, Rank #10: 86.829%, mAP:87.924%
Epoch: [2/50]| Loss: 0.16261


Total samples: 410, Rank #1: 57.073%, Rank #2: 67.561%, Rank #3: 71.707%, Rank #4: 74.634%, Rank #5: 76.585%, Rank #6: 79.512%, Rank #7: 81.220%, Rank #8: 83.659%, Rank #9: 84.878%, Rank #10: 86.098%, mAP:87.768%
Epoch: [3/50]| Loss: 0.12018


Total samples: 410, Rank #1: 53.415%, Rank #2: 64.146%, Rank #3: 70.732%, Rank #4: 75.122%, Rank #5: 78.049%, Rank #6: 79.512%, Rank #7: 81.707%, Rank #8: 81.951%, Rank #9: 81.951%, Rank #10: 83.659%, mAP:88.552%
Epoch: [4/50]| Loss: 0.10285


Total samples: 410, Rank #1: 45.366%, Rank #2: 57.317%, Rank #3: 65.854%, Rank #4: 71.463%, Rank #5: 74.634%, Rank #6: 77.561%, Rank #7: 79.512%, Rank #8: 80.732%, Rank #9: 82.439%, Rank #10: 83.415%, mAP:90.277%
Epoch: [5/50]| Loss: 0.08769


...

Total samples: 410, Rank #1: 2.439%, Rank #2: 10.488%, Rank #3: 13.171%, Rank #4: 15.610%, Rank #5: 18.049%, Rank #6: 23.902%, Rank #7: 25.610%, Rank #8: 28.537%, Rank #9: 30.732%, Rank #10: 32.683%, mAP:99.477%
Epoch: [46/50]| Loss: 0.03495


Total samples: 410, Rank #1: 2.439%, Rank #2: 13.415%, Rank #3: 15.854%, Rank #4: 19.268%, Rank #5: 21.707%, Rank #6: 25.610%, Rank #7: 26.585%, Rank #8: 29.756%, Rank #9: 31.951%, Rank #10: 33.415%, mAP:99.477%
Epoch: [47/50]| Loss: 0.03480


Total samples: 410, Rank #1: 14.634%, Rank #2: 20.976%, Rank #3: 23.902%, Rank #4: 26.098%, Rank #5: 30.244%, Rank #6: 30.732%, Rank #7: 32.195%, Rank #8: 33.659%, Rank #9: 34.146%, Rank #10: 35.610%, mAP:96.863%
Epoch: [48/50]| Loss: 0.03471


Total samples: 410, Rank #1: 2.439%, Rank #2: 13.415%, Rank #3: 19.024%, Rank #4: 22.439%, Rank #5: 25.610%, Rank #6: 28.537%, Rank #7: 29.268%, Rank #8: 30.976%, Rank #9: 32.683%, Rank #10: 32.927%, mAP:99.477%
Epoch: [49/50]| Loss: 0.03340


Total samples: 410, Rank #1: 2.439%, Rank #2: 10.976%, Rank #3: 13.902%, Rank #4: 16.585%, Rank #5: 19.024%, Rank #6: 22.195%, Rank #7: 23.902%, Rank #8: 27.073%, Rank #9: 30.732%, Rank #10: 31.951%, mAP:99.477%
Epoch: [50/50]| Loss: 0.03357


 ## Performance Evaluation

In [10]:
# Initialize the Siamese neural network models with the same architecture and parameters
m1 = BSN(backbone, img_channels, depth, img_w, img_h, n_classes).to(device)

# Load pre-trained model weights
m1.load_state_dict(torch.load('./output/models/best_'+backbone+'_state_r1.bin', map_location=device))

# Evaluate and print the performance of the model with the respective evaluation metric
print('Model w/ best R1:')
testing_step(m1, test_dataloader)

Model w/ best R1:


Total samples: 410, Rank #1: 64.634%, Rank #2: 72.439%, Rank #3: 76.585%, Rank #4: 79.268%, Rank #5: 80.732%, Rank #6: 82.683%, Rank #7: 84.878%, Rank #8: 86.341%, Rank #9: 87.073%, Rank #10: 87.805%, mAP:86.147%


(0.6463414634146342,
 0.8073170731707318,
 0.8780487804878049,
 np.float64(0.8614697595388653))